# Robustness and interpretability analysis for AstroLens GCNN

Adapts two analyses from the GCNN paper's reference implementation
([`GCNNMorphology`](https://github.com/snehjp2/GCNNMorphology),
`src/scripts/onepixelattack.py` and `src/scripts/latent_space_analysis.py`)
to `astrolens.models.gcnn.GCNN` and its registry:

- **One-pixel attack**: a black-box adversarial attack (Su et al., 2019) that
  searches for a single pixel whose color change flips the model's
  prediction, using differential evolution
  ([`scipy.optimize.differential_evolution`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html))
  instead of the reference implementation's `escnn`/CMA-ES-based search.
- **Latent-space analysis**: a t-SNE projection ([`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html),
  in place of the reference implementation's UMAP, to reuse the
  `scikit-learn` dependency already used for the train/val/test split) of the
  model's invariant embedding (`forward_features` + group pooling, before the
  classification head), colored by galaxy class.

Requires a checkpoint from `examples/gz10_gcnn_training.ipynb`
(`gcnn_d4.pt`) — run that notebook first.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [ ]:
!pip install -q datasets torchvision scikit-learn scipy

## Imports

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from PIL import Image
from scipy.optimize import differential_evolution
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

import astrolens

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the test split and the trained model

Same 70/10/20 split (`random_state=0`) as `gz10_gcnn_training.ipynb`, so `test_idx` reproduces the same held-out images the checkpoint was evaluated on.

In [ ]:
IMG_SIZE = 255
CHECKPOINT_PATH = "gcnn_d4.pt"
CLASS_NAMES = [
    "disturbed",
    "merging",
    "round_smooth",
    "in_between_round_smooth",
    "cigar_shaped_smooth",
    "barred_spiral",
    "unbarred_tight_spiral",
    "unbarred_loose_spiral",
    "edge_on_no_bulge",
    "edge_on_with_bulge",
]
NUM_CLASSES = len(CLASS_NAMES)

gz10 = load_dataset("UniverseTBD/mmu_gz10", split="train")
labels = gz10["gz10_label"]

train_idx, rest_idx = train_test_split(
    range(len(gz10)), train_size=0.7, stratify=labels, random_state=0
)
_, test_idx = train_test_split(
    rest_idx,
    train_size=1 / 3,
    stratify=[labels[i] for i in rest_idx],
    random_state=0,
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(IMG_SIZE),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)


class GZ10Dataset(Dataset):
    """Map-style wrapper around an index subset of the HF split, applying transform lazily."""

    def __init__(self, hf_split, indices, transform):
        self.hf_split = hf_split
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        example = self.hf_split[self.indices[i]]
        image = Image.open(io.BytesIO(example["rgb_image"]["bytes"])).convert("RGB")
        return self.transform(image), example["gz10_label"]


test_dataset = GZ10Dataset(gz10, test_idx, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=4)

model = astrolens.create_model(
    "gcnn_d4",
    img_size=IMG_SIZE,
    in_chans=3,
    num_classes=NUM_CLASSES,
).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

len(test_dataset)

## One-pixel attack

For each candidate image, `differential_evolution` searches over one pixel's `(x, y, r, g, b)` for the setting that minimizes the model's softmax probability on the image's true class — an attack succeeds if that probability drops below every other class's.

In [ ]:
def perturb(image: torch.Tensor, x) -> torch.Tensor:
    """Apply one pixel edit `x = (px, py, r, g, b)` (all in [0, 1]) to a copy of `image`."""
    _, h, w = image.shape
    px, py = round(x[0] * (w - 1)), round(x[1] * (h - 1))
    out = image.clone()
    out[:, py, px] = image.new_tensor(x[2:]) * 2 - 1  # [0, 1] -> normalized [-1, 1]
    return out


@torch.no_grad()
def true_class_prob(x, image: torch.Tensor, true_label: int) -> float:
    logits = model(perturb(image, x).unsqueeze(0).to(device))
    return torch.softmax(logits, dim=1)[0, true_label].item()


def one_pixel_attack(image: torch.Tensor, true_label: int, maxiter=30, popsize=15):
    bounds = [(0, 1)] * 5
    result = differential_evolution(
        true_class_prob,
        bounds,
        args=(image, true_label),
        maxiter=maxiter,
        popsize=popsize,
        tol=1e-4,
        seed=0,
        polish=False,
    )
    adversarial = perturb(image, result.x)
    return adversarial, result.x


N_ATTACK_IMAGES = 5

correct_images = []
with torch.no_grad():
    for images, batch_labels in test_loader:
        preds = model(images.to(device)).argmax(dim=1).cpu()
        for image, label, pred in zip(images, batch_labels, preds):
            if label == pred:
                correct_images.append((image, label.item()))
        if len(correct_images) >= N_ATTACK_IMAGES:
            break
correct_images = correct_images[:N_ATTACK_IMAGES]

attack_results = []
for image, true_label in correct_images:
    adversarial, pixel = one_pixel_attack(image, true_label)
    with torch.no_grad():
        adv_pred = model(adversarial.unsqueeze(0).to(device)).argmax(dim=1).item()
    attack_results.append((image, adversarial, true_label, adv_pred))

success_rate = sum(t != a for _, _, t, a in attack_results) / len(attack_results)
print(f"one-pixel attack success rate: {success_rate:.0%} ({len(attack_results)} images)")

In [ ]:
def show_image(ax, image: torch.Tensor, title: str):
    npimg = (image.cpu().numpy() * 0.5 + 0.5).clip(0, 1)  # undo Normalize(0.5, 0.5, 0.5)
    ax.imshow(np.transpose(npimg, (1, 2, 0)))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


fig, axes = plt.subplots(2, len(attack_results), figsize=(3 * len(attack_results), 6))
for i, (image, adversarial, true_label, adv_pred) in enumerate(attack_results):
    show_image(axes[0, i], image, f"original: {CLASS_NAMES[true_label]}")
    show_image(axes[1, i], adversarial, f"perturbed: {CLASS_NAMES[adv_pred]}")
fig.tight_layout()

## Latent-space analysis

`GCNN.forward_features` returns the regular-representation features before invariant group pooling; applying `model.gpool` and flattening gives the same rotation/reflection-invariant embedding the classification head consumes.

In [ ]:
@torch.no_grad()
def embed(images: torch.Tensor) -> torch.Tensor:
    features = model.forward_features(images.to(device))
    return model.gpool(features).flatten(1)


embeddings, embedding_labels = [], []
for images, batch_labels in test_loader:
    embeddings.append(embed(images).cpu())
    embedding_labels.append(batch_labels)
embeddings = torch.cat(embeddings).numpy()
embedding_labels = torch.cat(embedding_labels).numpy()

projection = TSNE(n_components=2, random_state=0, init="pca").fit_transform(embeddings)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    projection[:, 0], projection[:, 1], c=embedding_labels, cmap="tab10", s=6, alpha=0.7
)
handles, _ = scatter.legend_elements(num=NUM_CLASSES)
ax.legend(handles, CLASS_NAMES, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("t-SNE of GCNN invariant embeddings (test split)")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()